<a href="https://colab.research.google.com/github/MNawfal24/MachineLearning/blob/main/Minggu11/MNawfalMA_JS11_T01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import cv2
import glob
import os
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

Mounted at /content/drive


In [2]:
csv_path = '/content/drive/MyDrive/ML/voice.csv'

df_voice = pd.read_csv(csv_path)

# Separasi fitur dan label
X_voice = df_voice.drop('label', axis=1)
y_voice = df_voice['label']

# Encode label ('male'/'female' -> 1/0)
le = LabelEncoder()
y_voice_encoded = le.fit_transform(y_voice)

print("Shape data voice:", X_voice.shape)
print("Distribusi kelas:", pd.Series(y_voice_encoded).value_counts().to_dict())

Shape data voice: (3168, 20)
Distribusi kelas: {1: 1584, 0: 1584}


In [3]:
# Cell 3: Eksperimen SVM Voice (Kernel: Linear, Poly, RBF | Split: 70:30, 80:20)
kernels = ['linear', 'poly', 'rbf']
splits = [0.3, 0.2] # 0.3 -> 70:30, 0.2 -> 80:20

results_tugas1 = []

for test_size in splits:
    split_name = "70:30" if test_size == 0.3 else "80:20"

    X_train, X_test, y_train, y_test = train_test_split(
        X_voice, y_voice_encoded, test_size=test_size, random_state=42, stratify=y_voice_encoded
    )

    # Standardisasi fitur
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for kernel in kernels:
        svm = SVC(kernel=kernel, random_state=42)
        svm.fit(X_train_scaled, y_train)

        y_train_pred = svm.predict(X_train_scaled)
        y_test_pred = svm.predict(X_test_scaled)

        acc_train = accuracy_score(y_train, y_train_pred)
        acc_test = accuracy_score(y_test, y_test_pred)

        results_tugas1.append({
            'Split Ratio': split_name,
            'Kernel': kernel,
            'Train Accuracy': round(acc_train, 4),
            'Test Accuracy': round(acc_test, 4)
        })

# Tabulasi hasil
df_results1 = pd.DataFrame(results_tugas1)
print("=== HASIL EVALUASI TUGAS 1 (VOICE DATASET) ===")
display(df_results1)

=== HASIL EVALUASI TUGAS 1 (VOICE DATASET) ===


,Split Ratio,Kernel,Train Accuracy,Test Accuracy
0,70:30,linear,0.9720,0.9790
1,70:30,poly,0.9666,0.9600
2,70:30,rbf,0.9838,0.9832
3,80:20,linear,0.9767,0.9748
4,80:20,poly,0.9680,0.9558
5,80:20,rbf,0.9838,0.9826


In [4]:
# Cell 4: Load Citra & Ekstraksi Fitur Histogram (HSV)
# Ganti path folder dataset citra di Google Drive
dataset_img_dir = '/content/drive/MyDrive/ML/images/images'

def extract_histogram_features(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    # Resize standar
    img = cv2.resize(img, (256, 256))
    # Konversi ke HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    # Hitung histogram 3D (H, S, V)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8], [0, 180, 0, 256, 0, 256])
    # Normalisasi & ratakan
    cv2.normalize(hist, hist)
    return hist.flatten()

X_img = []
y_img = []

# Traversal folder dataset
for root, dirs, files in os.walk(dataset_img_dir):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            file_path = os.path.join(root, file)

            # Tentukan label dari nama folder ('day' -> 1, 'night' -> 0)
            if 'day' in file_path.lower():
                label = 1
            elif 'night' in file_path.lower():
                label = 0
            else:
                continue

            features = extract_histogram_features(file_path)
            if features is not None:
                X_img.append(features)
                y_img.append(label)

X_img = np.array(X_img)
y_img = np.array(y_img)

print("Fitur citra ter-ekstrak:", X_img.shape)
print("Distribusi label (1: Day, 0: Night):", np.bincount(y_img))

Fitur citra ter-ekstrak: (400, 512)
Distribusi label (1: Day, 0: Night): [200 200]


In [5]:
# Cell 5: Split 80:20 & Hyperparameter Tuning SVM RBF
# Split data 80:20
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_img, y_img, test_size=0.2, random_state=42, stratify=y_img
)

# Standardisasi fitur histogram
scaler_img = StandardScaler()
X_train_img_scaled = scaler_img.fit_transform(X_train_img)
X_test_img_scaled = scaler_img.transform(X_test_img)

# Parameter grid untuk SVM RBF
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
}

# GridSearchCV
grid_search = GridSearchCV(
    estimator=SVC(kernel='rbf', random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train_img_scaled, y_train_img)

best_model = grid_search.best_estimator_

# Evaluasi
y_pred_train = best_model.predict(X_train_img_scaled)
y_pred_test = best_model.predict(X_test_img_scaled)

print("=== HASIL TUGAS 2 (CITRA SIANG/MALAM) ===")
print("Parameter Terbaik:", grid_search.best_params_)
print(f"Akurasi Train : {accuracy_score(y_train_img, y_pred_train):.4f}")
print(f"Akurasi Test  : {accuracy_score(y_test_img, y_pred_test):.4f}\n")
print("Classification Report (Test Data):")
print(classification_report(y_test_img, y_pred_test, target_names=['Night', 'Day']))

=== HASIL TUGAS 2 (CITRA SIANG/MALAM) ===
Parameter Terbaik: {'C': 10, 'gamma': 0.001}
Akurasi Train : 1.0000
Akurasi Test  : 0.9750

Classification Report (Test Data):
              precision    recall  f1-score   support

       Night       0.95      1.00      0.98        40
         Day       1.00      0.95      0.97        40

    accuracy                           0.97        80
   macro avg       0.98      0.97      0.97        80
weighted avg       0.98      0.97      0.97        80

